[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/075.%20The%20VRP%20with%20Pickup%20and%20Delivery%20%28VRPPD%29/P75-Tier-8_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 75. The Vehicle Routing Problem with Pickup and Delivery (VRPPD)
## Tier 8: Value Alignment and Ethical Decision Support

### Purpose and scope

This notebook builds a small VRPPD solution with a first-fit constructive heuristic, then evaluates that solution using a transparent, illustrative ethical decision-support framework. It brings together route construction, principle-based assessments, stakeholder impact proxies, bias screening, decision records, and visual summaries.

**Inputs** are a pickup-and-delivery instance, vehicle count and capacity, and a distance matrix. **Outputs** are constructed routes and diagnostic scores, explanations, and plots. The route-construction method is a simple greedy heuristic: each request is assigned to the first vehicle with enough remaining aggregate assigned demand. This is not an exact VRPPD optimization algorithm, and it does not establish that a route plan is optimal.

The ethical components are educational prototypes, not validated legal, safety, fairness, privacy, or environmental assessments. Their scores are illustrative indicators defined by the code, not empirical findings. In particular, the framework reports recommendations and annotations; it does not change the routes or guarantee that a plan satisfies real-world ethical requirements.

### Modeling assumptions

- The distance matrix is symmetric, nonnegative, and measured in abstract distance units. It includes a start depot and a distinct end-depot node at the same modeled location.
- Each request has one pickup and one delivery. A request's pickup immediately precedes its delivery in the constructed route, so pickup-before-delivery precedence is respected.
- The heuristic limits the sum of request quantities assigned to each vehicle to its capacity. This conservative aggregate bound is sufficient for the constructed pickup-then-delivery sequences, but it can reject assignments that a more detailed load-feasibility model would allow.
- Travel distances are illustrative; time windows, travel time, vehicle heterogeneity, traffic, emissions, driver regulations, protected attributes, and real stakeholder feedback are outside the model.
- Principle weights and scoring thresholds are demonstration choices. They are not calibrated, externally validated, or a substitute for stakeholder governance.

### Method and learning sequence

1. Define the instance, ethical principles, and stakeholder groups.
2. Construct a feasible route plan with the first-fit heuristic.
3. Calculate illustrative principle-compliance and stakeholder-impact indicators.
4. Screen route workload for a limited imbalance signal and record whether the screen identifies it.
5. Record explanations and an accountability trail, analyze the decisions, and visualize the indicators.

A score is an output of the stated proxy, not proof that a decision is ethically acceptable. A detected imbalance is a prompt for review, not evidence of discrimination; a lack of detected imbalance is not evidence that bias is absent.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from collections import defaultdict
import copy
import random
import time

import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
random.seed(42)

print("Libraries imported; random seeds set for reproducible example data and simulated feedback.")

## 1. Define the VRPPD instance

A pickup-and-delivery request has two linked nodes: a pickup and its matching delivery. A vehicle must visit the pickup before the delivery. Here, pickup nodes are numbered from 1 through `num_pairs`, delivery nodes from `num_pairs + 1` through `2 * num_pairs`, and the start and end depots have separate node indices.

The signed demand vector stores positive quantities at pickups and matching negative quantities at deliveries. The constructive heuristic uses the positive pickup quantity when assigning a request. It also places each pickup immediately before its delivery, so the load returns to zero after each request.

In [ ]:
@dataclass
class VRPPDInstance:
    """Data and node sets for a pickup-and-delivery routing instance."""

    num_pairs: int
    num_vehicles: int
    vehicle_capacity: int
    distances: np.ndarray
    demands: List[int]

    def __post_init__(self) -> None:
        if self.num_pairs <= 0 or self.num_vehicles <= 0:
            raise ValueError("The instance must have at least one request and one vehicle.")
        if self.vehicle_capacity <= 0:
            raise ValueError("Vehicle capacity must be positive.")

        self.num_vertices = 2 * self.num_pairs + 2
        self.depot_start = 0
        self.depot_end = 2 * self.num_pairs + 1
        self.pickups = list(range(1, self.num_pairs + 1))
        self.deliveries = list(range(self.num_pairs + 1, 2 * self.num_pairs + 1))
        self.all_vertices = list(range(self.num_vertices))
        self.vehicles = list(range(self.num_vehicles))

        if self.distances.shape != (self.num_vertices, self.num_vertices):
            raise ValueError("The distance matrix dimensions do not match the instance.")
        if len(self.demands) != 2 * self.num_pairs:
            raise ValueError("The demand vector must contain one entry per pickup and delivery.")
        if np.any(self.distances < 0):
            raise ValueError("Distances must be nonnegative.")
        if not np.allclose(self.distances, self.distances.T):
            raise ValueError("This example assumes a symmetric distance matrix.")
        if not np.allclose(np.diag(self.distances), 0):
            raise ValueError("The distance matrix diagonal must be zero.")

        for pickup in self.pickups:
            delivery = pickup + self.num_pairs
            pickup_demand = self.demands[pickup - 1]
            delivery_demand = self.demands[delivery - 1]
            if pickup_demand <= 0 or delivery_demand != -pickup_demand:
                raise ValueError("Each delivery demand must negate its matching pickup demand.")
            if pickup_demand > self.vehicle_capacity:
                raise ValueError("A request quantity cannot exceed vehicle capacity.")


def create_ethical_instance() -> VRPPDInstance:
    """Create the reproducible, illustrative 12-request instance."""
    num_pairs = 12
    num_vehicles = 6
    vehicle_capacity = 8
    num_vertices = 2 * num_pairs + 2

    pickup_demands = np.random.randint(1, 4, num_pairs).tolist()
    delivery_demands = [-demand for demand in pickup_demands]
    demands = pickup_demands + delivery_demands

    # Generate symmetric abstract distances so every arc has a consistent cost.
    upper_triangle = np.zeros((num_vertices, num_vertices))
    for i in range(num_vertices):
        for j in range(i + 1, num_vertices):
            base_distance = 6 + abs(i - j) * 1.2 + np.random.uniform(-0.8, 0.8)
            upper_triangle[i, j] = max(1.0, base_distance)

    distances = upper_triangle + upper_triangle.T
    return VRPPDInstance(
        num_pairs=num_pairs,
        num_vehicles=num_vehicles,
        vehicle_capacity=vehicle_capacity,
        distances=distances,
        demands=demands,
    )


ethical_instance = create_ethical_instance()
print("Illustrative VRPPD instance created:")
print(f"- Pickup-delivery pairs: {ethical_instance.num_pairs}")
print(f"- Vehicles: {ethical_instance.num_vehicles}")
print(f"- Vehicle capacity: {ethical_instance.vehicle_capacity}")
print(f"- Total vertices, including both depot nodes: {ethical_instance.num_vertices}")
print(f"- Sum of positive pickup quantities: {sum(ethical_instance.demands[:ethical_instance.num_pairs])}")

## 2. Represent principles, stakeholders, and decisions

The framework stores principles and stakeholder groups as named records. Principle weights and stakeholder weights express the example's declared priorities; they do not determine route assignments in this notebook. An `EthicalDecision` keeps the initial route plan, an annotated copy, scores, explanations, screening results, and an audit trail so that the evaluation can be inspected.

In [ ]:
@dataclass
class EthicalPrinciple:
    """A principle and its illustrative weight and evaluation indicators."""

    name: str
    description: str
    weight: float
    constraints: List[str]
    metrics: List[str]


@dataclass
class Stakeholder:
    """A stakeholder group and its stated concerns."""

    name: str
    category: str
    weight: float
    concerns: List[str]
    satisfaction_score: float


@dataclass
class EthicalDecision:
    """A recorded diagnostic evaluation of one route plan."""

    decision_id: str
    decision_type: str
    original_solution: Dict[str, Any]
    ethical_solution: Dict[str, Any]
    principle_compliance: Dict[str, float]
    stakeholder_impact: Dict[str, float]
    ethical_reasoning: List[str]
    bias_detected: List[str]
    bias_mitigated: List[str]
    transparency_score: float
    accountability_trail: List[Dict[str, Any]]
    timestamp: float

## 3. Build the ethical decision-support components

A compliance score is a numerical proxy used to summarize a specified rule; it is not a formal guarantee of compliance. In this example, fairness is assessed from the spread of active-route distances, safety uses illustrative distance and stop-count thresholds, and sustainability uses total distance as an emissions proxy. Privacy and accountability scores are explicitly assumed rather than measured from data.

The bias screen is deliberately narrow: it flags high variation in active-route distances as a workload-imbalance signal. It cannot establish discrimination because the instance contains no protected-group information or causal analysis. The framework does not change route stops or distances: changing either without rebuilding and revalidating a route could invalidate the VRPPD plan. A detected signal therefore remains a review item, not a claimed mitigation.

In [ ]:
class ConstraintEngine:
    """Calculate illustrative indicators and annotate, rather than rewrite, routes."""

    def __init__(self) -> None:
        self.constraint_handlers = {
            "fairness": self._apply_fairness_constraints,
            "safety": self._apply_safety_constraints,
            "sustainability": self._apply_sustainability_constraints,
            "privacy": self._apply_privacy_constraints,
            "accountability": self._apply_accountability_constraints,
        }

    def check_principle_compliance(
        self,
        principle: EthicalPrinciple,
        solution: Dict[str, Any],
        context: Dict[str, Any],
    ) -> float:
        """Return the score from the matching illustrative indicator."""
        handler = self.constraint_handlers.get(principle.name.lower())
        if handler is None:
            return 0.5
        return handler(solution, context, check_only=True)

    def apply_constraints(
        self,
        solution: Dict[str, Any],
        principle_compliance: Dict[str, float],
    ) -> Dict[str, Any]:
        """Return an annotated copy and preserve the feasible route plan."""
        ethical_solution = copy.deepcopy(solution)
        review_items = [
            f"Review the {name} indicator (score {score:.2f}); no route change was applied."
            for name, score in principle_compliance.items()
            if score < 0.6
        ]
        ethical_solution["ethical_review_items"] = review_items
        return ethical_solution

    @staticmethod
    def _active_routes(solution: Dict[str, Any]) -> List[Dict[str, Any]]:
        return [
            route for route in solution.get("routes", [])
            if len(route.get("stops", [])) > 2
        ]

    def _apply_fairness_constraints(
        self, solution: Dict[str, Any], context: Dict[str, Any], check_only: bool = False
    ) -> float:
        """Use route-distance balance as a workload proxy, not a fairness proof."""
        del context, check_only
        routes = self._active_routes(solution)
        if len(routes) < 2:
            return 0.8
        lengths = np.array([route.get("distance", 0.0) for route in routes])
        mean_length = float(np.mean(lengths))
        if mean_length == 0:
            return 1.0
        coefficient_of_variation = float(np.std(lengths) / mean_length)
        return float(np.clip(1.0 - coefficient_of_variation, 0.0, 1.0))

    def _apply_safety_constraints(
        self, solution: Dict[str, Any], context: Dict[str, Any], check_only: bool = False
    ) -> float:
        """Apply the example's declared distance and stop-count thresholds."""
        del context, check_only
        routes = self._active_routes(solution)
        if not routes:
            return 0.5
        violations = sum(
            route.get("distance", 0.0) > 100
            or len(route.get("stops", [])) - 2 > 15
            for route in routes
        )
        return float(1.0 - violations / len(routes))

    def _apply_sustainability_constraints(
        self, solution: Dict[str, Any], context: Dict[str, Any], check_only: bool = False
    ) -> float:
        """Use total route distance as an uncalibrated environmental proxy."""
        del context, check_only
        total_distance = sum(
            route.get("distance", 0.0) for route in solution.get("routes", [])
        )
        return float(max(0.0, 1.0 - total_distance / 300.0))

    def _apply_privacy_constraints(
        self, solution: Dict[str, Any], context: Dict[str, Any], check_only: bool = False
    ) -> float:
        """Return an assumed demonstration score; no privacy audit is performed."""
        del solution, context, check_only
        return 0.85

    def _apply_accountability_constraints(
        self, solution: Dict[str, Any], context: Dict[str, Any], check_only: bool = False
    ) -> float:
        """Return an assumed score; audit records are illustrative, not regulatory."""
        del solution, context, check_only
        return 0.90


class BiasDetector:
    """Screen for route-distance imbalance without claiming to prove bias."""

    def __init__(self) -> None:
        self.bias_patterns = {
            "distance_bias": "A route-distance imbalance signal that merits review."
        }

    def detect_biases(
        self, solution: Dict[str, Any], context: Dict[str, Any]
    ) -> List[str]:
        del context
        routes = ConstraintEngine._active_routes(solution)
        if len(routes) < 2:
            return []
        distances = np.array([route.get("distance", 0.0) for route in routes])
        mean_distance = float(np.mean(distances))
        if mean_distance > 0 and float(np.std(distances) / mean_distance) > 0.5:
            return ["distance_bias"]
        return []

    def mitigate_biases(
        self, solution: Dict[str, Any], detected_biases: List[str]
    ) -> List[str]:
        """Do not claim mitigation when no validated route reassignment is made."""
        del solution, detected_biases
        return []


class TransparencySystem:
    """Summarize score coverage and compliance disclosure."""

    def calculate_transparency_score(
        self,
        solution: Dict[str, Any],
        principle_compliance: Dict[str, float],
        stakeholder_impact: Dict[str, float],
    ) -> float:
        del solution
        if not principle_compliance:
            return 0.0
        average_compliance = float(np.mean(list(principle_compliance.values())))
        impact_coverage = min(1.0, len(stakeholder_impact) / 4.0)
        return float(np.clip(0.6 * average_compliance + 0.4 * impact_coverage, 0.0, 1.0))


class AccountabilityFramework:
    """Create an illustrative sequence of decision-evaluation records."""

    def create_trail(
        self,
        decision_id: str,
        solution: Dict[str, Any],
        principle_compliance: Dict[str, float],
        stakeholder_impact: Dict[str, float],
    ) -> List[Dict[str, Any]]:
        del solution
        actions = [
            ("decision_initiated", "Ethical evaluation initiated."),
            ("principle_evaluation", f"Principle indicators: {principle_compliance}"),
            ("stakeholder_assessment", f"Stakeholder proxies: {stakeholder_impact}"),
            ("decision_recorded", "Diagnostic decision record created."),
        ]
        start_time = time.time()
        return [
            {
                "timestamp": start_time + index * 0.001,
                "action": action,
                "decision_id": decision_id,
                "responsible_party": "illustrative_ethical_framework",
                "details": details,
            }
            for index, (action, details) in enumerate(actions)
        ]


class StakeholderFeedbackSystem:
    """Store feedback records, including clearly labeled simulated feedback."""

    def __init__(self) -> None:
        self.feedback_history: List[Dict[str, Any]] = []
        self.satisfaction_trends = defaultdict(list)

    def collect_feedback(
        self, stakeholder: str, decision_id: str, satisfaction: float, comments: str
    ) -> Dict[str, Any]:
        if not 0.0 <= satisfaction <= 1.0:
            raise ValueError("Satisfaction must be between 0 and 1.")
        feedback = {
            "timestamp": time.time(),
            "stakeholder": stakeholder,
            "decision_id": decision_id,
            "satisfaction": satisfaction,
            "comments": comments,
        }
        self.feedback_history.append(feedback)
        self.satisfaction_trends[stakeholder].append(satisfaction)
        return feedback

## 4. Define the principles and stakeholder groups

The weights below sum to one within each framework. They document the example's priorities, but this implementation does not combine them into a route-optimization objective. The route heuristic is constructed first; ethical indicators are calculated afterward.

In [ ]:
class EthicalConstitution:
    """Collect the example's principles, stakeholders, and evaluation tools."""

    def __init__(self) -> None:
        self.principles = self._define_ethical_principles()
        self.stakeholders = self._define_stakeholders()
        self.constraint_engine = ConstraintEngine()
        self.bias_detector = BiasDetector()
        self.transparency_system = TransparencySystem()
        self.accountability_framework = AccountabilityFramework()
        self.feedback_system = StakeholderFeedbackSystem()
        self.decision_counter = 0

    def _define_ethical_principles(self) -> Dict[str, EthicalPrinciple]:
        return {
            "fairness": EthicalPrinciple(
                name="Fairness",
                description="Consider equitable treatment and route workload balance.",
                weight=0.25,
                constraints=["review_route_workload_balance"],
                metrics=["route_distance_balance_proxy"],
            ),
            "safety": EthicalPrinciple(
                name="Safety",
                description="Screen route length and stop count using example thresholds.",
                weight=0.30,
                constraints=["review_long_routes", "review_high_stop_counts"],
                metrics=["threshold_screen_score"],
            ),
            "sustainability": EthicalPrinciple(
                name="Sustainability",
                description="Use total distance as an illustrative environmental proxy.",
                weight=0.20,
                constraints=["review_total_distance"],
                metrics=["distance_proxy_score"],
            ),
            "privacy": EthicalPrinciple(
                name="Privacy",
                description="Privacy practices are not measured in this routing example.",
                weight=0.15,
                constraints=["privacy_review_required"],
                metrics=["assumed_demonstration_score"],
            ),
            "accountability": EthicalPrinciple(
                name="Accountability",
                description="Record the evaluation steps for inspection.",
                weight=0.10,
                constraints=["record_evaluation_steps"],
                metrics=["assumed_demonstration_score"],
            ),
        }

    def _define_stakeholders(self) -> Dict[str, Stakeholder]:
        return {
            "customers": Stakeholder(
                name="Customers",
                category="customer",
                weight=0.45,
                concerns=["timely service", "service quality", "privacy", "accessibility"],
                satisfaction_score=0.75,
            ),
            "drivers": Stakeholder(
                name="Drivers",
                category="driver",
                weight=0.25,
                concerns=["workload balance", "working conditions", "safety"],
                satisfaction_score=0.70,
            ),
            "environment": Stakeholder(
                name="environment",
                category="environment",
                weight=0.20,
                concerns=["emissions", "air quality", "resource efficiency"],
                satisfaction_score=0.60,
            ),
            "business": Stakeholder(
                name="Business",
                category="business",
                weight=0.10,
                concerns=["operational efficiency", "service quality", "accountability"],
                satisfaction_score=0.80,
            ),
        }

    def evaluate_solution_ethically(
        self, solution: Dict[str, Any], context: Dict[str, Any]
    ) -> EthicalDecision:
        """Evaluate a route plan and preserve its route decisions."""
        self.decision_counter += 1
        decision_id = f"ethical_decision_{self.decision_counter:03d}"

        principle_compliance = {
            name: self.constraint_engine.check_principle_compliance(
                principle, solution, context
            )
            for name, principle in self.principles.items()
        }
        stakeholder_impact = self._evaluate_stakeholder_impacts(solution, context)
        biases_detected = self.bias_detector.detect_biases(solution, context)
        biases_mitigated = self.bias_detector.mitigate_biases(
            solution, biases_detected
        )
        ethical_reasoning = self._generate_ethical_reasoning(
            principle_compliance, stakeholder_impact, biases_detected
        )
        transparency_score = self.transparency_system.calculate_transparency_score(
            solution, principle_compliance, stakeholder_impact
        )
        accountability_trail = self.accountability_framework.create_trail(
            decision_id, solution, principle_compliance, stakeholder_impact
        )
        ethical_solution = self.constraint_engine.apply_constraints(
            solution, principle_compliance
        )

        return EthicalDecision(
            decision_id=decision_id,
            decision_type="route_plan_diagnostic_evaluation",
            original_solution=copy.deepcopy(solution),
            ethical_solution=ethical_solution,
            principle_compliance=principle_compliance,
            stakeholder_impact=stakeholder_impact,
            ethical_reasoning=ethical_reasoning,
            bias_detected=biases_detected,
            bias_mitigated=biases_mitigated,
            transparency_score=transparency_score,
            accountability_trail=accountability_trail,
            timestamp=time.time(),
        )

    def _evaluate_stakeholder_impacts(
        self, solution: Dict[str, Any], context: Dict[str, Any]
    ) -> Dict[str, float]:
        """Calculate illustrative indicators, not measured satisfaction."""
        del context
        total_distance = sum(
            route.get("distance", 0.0) for route in solution.get("routes", [])
        )
        num_requests = max(1, len(solution.get("requests", [])))
        customer_proxy = max(0.0, 1.0 - (total_distance / num_requests) / 50.0)

        active_routes = ConstraintEngine._active_routes(solution)
        if len(active_routes) > 1:
            route_lengths = np.array(
                [route.get("distance", 0.0) for route in active_routes]
            )
            mean_length = float(np.mean(route_lengths))
            driver_proxy = (
                1.0
                if mean_length == 0
                else max(0.0, 1.0 - float(np.var(route_lengths)) / mean_length**2)
            )
        else:
            driver_proxy = 0.5

        environment_proxy = max(0.0, 1.0 - total_distance / 200.0)
        business_proxy = max(0.0, 1.0 - total_distance / 150.0)
        return {
            "customers": float(np.clip(customer_proxy, 0.0, 1.0)),
            "drivers": float(np.clip(driver_proxy, 0.0, 1.0)),
            "environment": float(np.clip(environment_proxy, 0.0, 1.0)),
            "business": float(np.clip(business_proxy, 0.0, 1.0)),
        }

    @staticmethod
    def _generate_ethical_reasoning(
        principle_compliance: Dict[str, float],
        stakeholder_impact: Dict[str, float],
        biases_detected: List[str],
    ) -> List[str]:
        reasoning = []
        for principle, score in principle_compliance.items():
            if score >= 0.8:
                label = "higher"
            elif score >= 0.6:
                label = "moderate"
            else:
                label = "lower; review the indicator and its assumptions"
            reasoning.append(f"{principle.title()} indicator: {score:.2f} ({label}).")

        for stakeholder, score in stakeholder_impact.items():
            reasoning.append(
                f"{stakeholder.title()} proxy: {score:.2f}; this is not measured satisfaction."
            )

        if biases_detected:
            reasoning.append(
                "A route-distance imbalance signal was flagged for review; this does not prove discrimination."
            )
        else:
            reasoning.append(
                "The limited route-distance screen flagged no imbalance; this does not establish that bias is absent."
            )
        return reasoning


print("Ethical decision-support structure defined.")

## 5. Construct and evaluate route plans

The constructive heuristic processes requests in their given order. For each request, it scans vehicles in order and assigns the request to the first route whose current assigned quantity plus the request quantity does not exceed capacity. The route then appends the pickup followed immediately by its matching delivery. Once assignments are complete, the distance calculation includes every leg from the start depot through the stops to the end depot.

This first-fit rule is straightforward and reproducible, but it does not compare alternative assignments or search for a minimum-distance plan. The ethical evaluation follows route construction; it is diagnostic and does not feed back into the heuristic.

In [ ]:
class EthicalVRPPDOptimizer:
    """Construct a greedy VRPPD plan and evaluate it with diagnostic proxies."""

    def __init__(self, instance: VRPPDInstance):
        self.instance = instance
        self.constitution = EthicalConstitution()
        self.optimization_history: List[EthicalDecision] = []
        self.performance_metrics = {
            "total_decisions": 0,
            "avg_principle_compliance": 0.0,
            "avg_stakeholder_proxy": 0.0,
            "bias_signal_rate": 0.0,
            "transparency_score": 0.0,
            "ethical_effectiveness": 0.0,
        }

    def generate_initial_solution(self) -> Dict[str, Any]:
        """Assign each request to the first vehicle with remaining capacity."""
        routes = [
            {
                "vehicle_id": vehicle_id,
                "stops": [self.instance.depot_start],
                "distance": 0.0,
                "load": 0,
            }
            for vehicle_id in self.instance.vehicles
        ]

        for pickup in self.instance.pickups:
            delivery = pickup + self.instance.num_pairs
            demand = self.instance.demands[pickup - 1]
            for route in routes:
                if route["load"] + demand <= self.instance.vehicle_capacity:
                    route["stops"].extend([pickup, delivery])
                    route["load"] += demand
                    break
            else:
                raise ValueError(
                    f"No vehicle has sufficient remaining capacity for request {pickup}."
                )

        for route in routes:
            route["stops"].append(self.instance.depot_end)
            route["distance"] = sum(
                self.instance.distances[origin, destination]
                for origin, destination in zip(route["stops"], route["stops"][1:])
            )

        solution = {
            "routes": routes,
            "requests": list(self.instance.pickups),
            "total_distance": sum(route["distance"] for route in routes),
            "vehicles_used": sum(route["load"] > 0 for route in routes),
        }
        self._validate_solution(solution)
        return solution

    def _validate_solution(self, solution: Dict[str, Any]) -> None:
        """Check request coverage, precedence, capacity, and reported distance."""
        visited_pickups = []
        visited_deliveries = []
        calculated_total_distance = 0.0

        for route in solution["routes"]:
            stops = route["stops"]
            if stops[0] != self.instance.depot_start or stops[-1] != self.instance.depot_end:
                raise AssertionError("Every route must start and end at the designated depots.")
            if route["load"] > self.instance.vehicle_capacity:
                raise AssertionError("A route exceeds the aggregate assigned-demand capacity.")

            positions = {node: index for index, node in enumerate(stops)}
            for pickup in self.instance.pickups:
                delivery = pickup + self.instance.num_pairs
                if pickup in positions:
                    if delivery not in positions or positions[pickup] >= positions[delivery]:
                        raise AssertionError("A pickup must precede its matching delivery.")
                    visited_pickups.append(pickup)
                    visited_deliveries.append(delivery)

            route_distance = sum(
                self.instance.distances[origin, destination]
                for origin, destination in zip(stops, stops[1:])
            )
            if not np.isclose(route_distance, route["distance"]):
                raise AssertionError("The reported route distance does not match its legs.")
            calculated_total_distance += route_distance

        if sorted(visited_pickups) != self.instance.pickups:
            raise AssertionError("Every pickup must appear exactly once across the routes.")
        expected_deliveries = [
            pickup + self.instance.num_pairs for pickup in self.instance.pickups
        ]
        if sorted(visited_deliveries) != expected_deliveries:
            raise AssertionError("Every delivery must appear exactly once across the routes.")
        if not np.isclose(calculated_total_distance, solution["total_distance"]):
            raise AssertionError("The reported total distance must equal the route-distance sum.")

    def optimize_ethically(
        self, context: Optional[Dict[str, Any]] = None
    ) -> EthicalDecision:
        """Construct a plan, then evaluate it without changing its routes."""
        if context is None:
            context = {}
        initial_solution = self.generate_initial_solution()
        decision = self.constitution.evaluate_solution_ethically(
            initial_solution, context
        )
        self._update_performance_metrics(decision)
        self.optimization_history.append(decision)
        return decision

    def _update_performance_metrics(self, decision: EthicalDecision) -> None:
        """Maintain running averages of diagnostic indicators."""
        count = self.performance_metrics["total_decisions"] + 1
        self.performance_metrics["total_decisions"] = count
        average_compliance = float(np.mean(list(decision.principle_compliance.values())))
        average_stakeholder_proxy = float(np.mean(list(decision.stakeholder_impact.values())))
        bias_signal = float(bool(decision.bias_detected))

        for key, value in [
            ("avg_principle_compliance", average_compliance),
            ("avg_stakeholder_proxy", average_stakeholder_proxy),
            ("bias_signal_rate", bias_signal),
            ("transparency_score", decision.transparency_score),
        ]:
            previous_average = self.performance_metrics[key]
            self.performance_metrics[key] = previous_average + (value - previous_average) / count

        # This composite summarizes proxies; it is not a validated ethical-quality measure.
        self.performance_metrics["ethical_effectiveness"] = (
            0.4 * self.performance_metrics["avg_principle_compliance"]
            + 0.3 * self.performance_metrics["avg_stakeholder_proxy"]
            + 0.3 * self.performance_metrics["transparency_score"]
        )

    def run_ethical_optimization_session(
        self, num_decisions: int = 30
    ) -> List[EthicalDecision]:
        """Repeat evaluation under labeled context scenarios."""
        if num_decisions <= 0:
            raise ValueError("The session must contain at least one decision.")

        print(f"Running an illustrative evaluation session with {num_decisions} decisions...")
        session_decisions = []
        for decision_number in range(num_decisions):
            context = {
                "decision_number": decision_number + 1,
                "demands": self.instance.demands,
                "time_of_day": random.choice(["morning", "afternoon", "evening"]),
                "weather_condition": random.choice(["clear", "rain", "snow", "fog"]),
                "traffic_level": random.choice(["light", "moderate", "heavy"]),
            }
            decision = self.optimize_ethically(context)
            session_decisions.append(decision)

            if (decision_number + 1) % 10 == 0:
                print(
                    f"  Decision {decision_number + 1}: "
                    f"proxy composite {self.performance_metrics['ethical_effectiveness']:.2f}, "
                    f"principle indicators {self.performance_metrics['avg_principle_compliance']:.2f}, "
                    f"stakeholder proxies {self.performance_metrics['avg_stakeholder_proxy']:.2f}"
                )

        print("Evaluation session completed.")
        return session_decisions


print("Greedy route constructor and diagnostic evaluator defined.")

## 6. Run the evaluation session

The session repeats the same deterministic route construction while attaching different illustrative context labels. Those labels are recorded but do not alter the routes or the scores in this implementation. Repeating the evaluation demonstrates how records and running averages are accumulated; it is not a statistical experiment and does not create independent route plans.

In [ ]:
ethical_optimizer = EthicalVRPPDOptimizer(ethical_instance)

print("Ethical decision-support evaluator initialized:")
print(f"- Principles: {len(ethical_optimizer.constitution.principles)}")
print(f"- Stakeholder groups: {len(ethical_optimizer.constitution.stakeholders)}")
print(f"- Pickup-delivery pairs: {ethical_instance.num_pairs}")
print(f"- Vehicles: {ethical_instance.num_vehicles}")

print("\nPrinciples and stated weights:")
for principle in ethical_optimizer.constitution.principles.values():
    print(f"- {principle.name}: {principle.description} (weight: {principle.weight:.2f})")

print("\nStakeholder groups and stated weights:")
for stakeholder in ethical_optimizer.constitution.stakeholders.values():
    print(
        f"- {stakeholder.name}: {stakeholder.category} "
        f"(weight: {stakeholder.weight:.2f}; stated baseline: {stakeholder.satisfaction_score:.2f})"
    )

ethical_decisions = ethical_optimizer.run_ethical_optimization_session(num_decisions=30)

## 7. Analyze the diagnostic results

The analysis averages the indicators across recorded decisions and counts detected signals and recorded mitigations separately. Because the session reuses the same instance and route heuristic, these averages summarize repeated evaluations of one constructed plan. A zero mitigation rate would mean that this prototype took no mitigation action; it would not mean that the screening signal was absent or that mitigation failed.

In [ ]:
def analyze_ethical_results(decisions: List[EthicalDecision]) -> Dict[str, Any]:
    """Aggregate principle indicators, stakeholder proxies, and screening records."""
    if not decisions:
        raise ValueError("At least one decision is required for analysis.")

    principle_data = defaultdict(list)
    stakeholder_data = defaultdict(list)
    for decision in decisions:
        for principle, score in decision.principle_compliance.items():
            principle_data[principle].append(score)
        for stakeholder, score in decision.stakeholder_impact.items():
            stakeholder_data[stakeholder].append(score)

    average_principle_compliance = {
        principle: float(np.mean(scores))
        for principle, scores in principle_data.items()
    }
    average_stakeholder_impact = {
        stakeholder: float(np.mean(scores))
        for stakeholder, scores in stakeholder_data.items()
    }

    total_biases_detected = sum(len(decision.bias_detected) for decision in decisions)
    total_biases_mitigated = sum(len(decision.bias_mitigated) for decision in decisions)
    mitigation_rate = (
        total_biases_mitigated / total_biases_detected
        if total_biases_detected
        else 0.0
    )
    transparency_scores = [decision.transparency_score for decision in decisions]
    ethical_scores = [
        0.4 * float(np.mean(list(decision.principle_compliance.values())))
        + 0.3 * float(np.mean(list(decision.stakeholder_impact.values())))
        + 0.3 * decision.transparency_score
        for decision in decisions
    ]

    return {
        "total_decisions": len(decisions),
        "avg_principle_compliance": average_principle_compliance,
        "avg_stakeholder_impact": average_stakeholder_impact,
        "total_biases_detected": total_biases_detected,
        "total_biases_mitigated": total_biases_mitigated,
        "bias_mitigation_rate": mitigation_rate,
        "avg_transparency": float(np.mean(transparency_scores)),
        "avg_ethical_score": float(np.mean(ethical_scores)),
        "principle_compliance_data": principle_data,
        "stakeholder_impact_data": stakeholder_data,
        "transparency_scores": transparency_scores,
        "ethical_scores": ethical_scores,
    }


ethical_analysis = analyze_ethical_results(ethical_decisions)
print("\nIllustrative VRPPD diagnostic summary")
print(f"Evaluations: {ethical_analysis['total_decisions']}")
print(f"Mean proxy composite: {ethical_analysis['avg_ethical_score']:.2f}")
print(f"Mean transparency indicator: {ethical_analysis['avg_transparency']:.2f}")
print(
    "Detected screening signals: "
    f"{ethical_analysis['total_biases_detected']} across all recorded evaluations"
)
print(
    "Recorded mitigation actions: "
    f"{ethical_analysis['total_biases_mitigated']} "
    f"(rate: {ethical_analysis['bias_mitigation_rate']:.1%})"
)

print("\nMean principle indicators:")
for principle, score in ethical_analysis["avg_principle_compliance"].items():
    print(f"- {principle}: {score:.2f}")

print("\nMean stakeholder proxies (not measured satisfaction):")
for stakeholder, score in ethical_analysis["avg_stakeholder_impact"].items():
    print(f"- {stakeholder}: {score:.2f}")

## 8. Visualize the indicators

The four panels show principle indicators, stakeholder proxies, the distribution of the composite proxy score, and the counts of screening signals and recorded mitigation actions. Axis labels and legends identify what is being plotted. These plots describe this example's calculations; they do not validate the indicators or establish causation.

In [ ]:
def visualize_ethical_results(
    decisions: List[EthicalDecision], analysis: Dict[str, Any]
):
    """Plot the evaluation indicators and their stated limitations."""
    if not decisions:
        raise ValueError("At least one decision is required for visualization.")

    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "VRPPD Ethical Decision-Support Indicators (Illustrative Proxies)",
        fontsize=15,
        fontweight="bold",
    )

    ax1.set_title("Principle Indicators Across Evaluations")
    for principle, values in analysis["principle_compliance_data"].items():
        ax1.plot(
            range(1, len(values) + 1),
            values,
            marker="o",
            linewidth=1.5,
            label=principle.replace("_", " ").title(),
        )
    ax1.set_xlabel("Evaluation number")
    ax1.set_ylabel("Illustrative indicator (0–1)")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=8)

    ax2.set_title("Mean Stakeholder Proxies")
    stakeholders = list(analysis["avg_stakeholder_impact"])
    impacts = [analysis["avg_stakeholder_impact"][name] for name in stakeholders]
    colors = ["steelblue", "seagreen", "goldenrod", "indianred"]
    bars = ax2.bar(stakeholders, impacts, color=colors[:len(stakeholders)], edgecolor="black")
    ax2.set_ylabel("Illustrative proxy (0–1)")
    ax2.set_ylim(0, 1.12)
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, impact in zip(bars, impacts):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{impact:.2f}",
            ha="center",
        )

    ax3.set_title("Composite Proxy Score Distribution")
    scores = analysis["ethical_scores"]
    ax3.hist(scores, bins=min(15, max(1, len(scores))), color="lightsteelblue", edgecolor="black")
    ax3.axvline(
        np.mean(scores),
        color="darkred",
        linestyle="--",
        linewidth=2,
        label=f"Mean: {np.mean(scores):.2f}",
    )
    ax3.set_xlabel("Composite proxy score")
    ax3.set_ylabel("Evaluation count")
    ax3.grid(True, alpha=0.3)
    ax3.legend()

    ax4.set_title("Screening Signals and Recorded Actions")
    bias_counts = defaultdict(int)
    mitigation_counts = defaultdict(int)
    for decision in decisions:
        for bias in decision.bias_detected:
            bias_counts[bias] += 1
        for bias in decision.bias_mitigated:
            mitigation_counts[bias] += 1

    bias_types = sorted(set(bias_counts) | set(mitigation_counts))
    if bias_types:
        positions = np.arange(len(bias_types))
        width = 0.35
        ax4.bar(
            positions - width / 2,
            [bias_counts[name] for name in bias_types],
            width,
            label="Signals detected",
            color="indianred",
        )
        ax4.bar(
            positions + width / 2,
            [mitigation_counts[name] for name in bias_types],
            width,
            label="Actions recorded",
            color="seagreen",
        )
        ax4.set_xticks(positions)
        ax4.set_xticklabels(
            [name.replace("_", " ").title() for name in bias_types], rotation=25, ha="right"
        )
        ax4.legend()
    else:
        ax4.text(0.5, 0.5, "No screening signals recorded", ha="center", va="center")
        ax4.set_xticks([])
    ax4.set_ylabel("Evaluation count")
    ax4.grid(True, axis="y", alpha=0.3)

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_ethical_results(ethical_decisions, ethical_analysis)
print("Plots display illustrative indicators, not validated ethical outcomes.")

## 9. Inspect one decision and collect simulated feedback

A decision record makes the evaluation inspectable: it contains the route plan, the proxy scores, the screening result, and the sequence of recorded steps. The feedback generated below is simulated with the notebook's seeded random generator. It is not feedback from actual customers, drivers, environmental representatives, or businesses, and it does not update the route plan or calibrate the scoring rules.

In [ ]:
def demonstrate_ethical_reasoning() -> EthicalDecision:
    """Print a decision record and collect explicitly simulated feedback."""
    context = {
        "demands": ethical_instance.demands,
        "time_of_day": "morning",
        "weather_condition": "clear",
        "traffic_level": "moderate",
    }
    decision = ethical_optimizer.optimize_ethically(context)

    print("\nDecision record")
    print(f"Decision ID: {decision.decision_id}")
    print(f"Decision type: {decision.decision_type}")
    print(f"Transparency indicator: {decision.transparency_score:.2f}")

    print("\nPrinciple indicators:")
    for principle, score in decision.principle_compliance.items():
        print(f"- {principle}: {score:.2f}")

    print("\nStakeholder proxies (not observed satisfaction):")
    for stakeholder, score in decision.stakeholder_impact.items():
        print(f"- {stakeholder}: {score:.2f}")

    print("\nReasoning notes:")
    for number, reasoning in enumerate(decision.ethical_reasoning, start=1):
        print(f"{number}. {reasoning}")

    print("\nScreening result:")
    if decision.bias_detected:
        print(f"Signals flagged for review: {', '.join(decision.bias_detected)}")
    else:
        print("No signal from the limited route-distance screen.")
    print(f"Mitigation actions recorded: {len(decision.bias_mitigated)}")

    print("\nIllustrative accountability trail:")
    for number, entry in enumerate(decision.accountability_trail, start=1):
        print(f"{number}. {entry['action'].replace('_', ' ').title()}: {entry['details']}")

    print("\nSimulated stakeholder feedback (not real survey data):")
    for stakeholder_name in ethical_optimizer.constitution.stakeholders:
        satisfaction = random.uniform(0.6, 0.95)
        feedback = ethical_optimizer.constitution.feedback_system.collect_feedback(
            stakeholder_name,
            decision.decision_id,
            satisfaction,
            "Synthetic example feedback; not collected from a stakeholder.",
        )
        print(f"- {stakeholder_name}: simulated score {feedback['satisfaction']:.2f}")

    average_compliance = float(np.mean(list(decision.principle_compliance.values())))
    average_proxy = float(np.mean(list(decision.stakeholder_impact.values())))
    composite_proxy = 0.4 * average_compliance + 0.3 * average_proxy + 0.3 * decision.transparency_score
    print("\nComposite proxy summary")
    print(f"Mean principle indicator: {average_compliance:.2f}")
    print(f"Mean stakeholder proxy: {average_proxy:.2f}")
    print(f"Transparency indicator: {decision.transparency_score:.2f}")
    print(f"Illustrative composite: {composite_proxy:.2f}")
    print("Interpret this composite only as the weighted summary defined in this notebook.")
    return decision


demonstration_decision = demonstrate_ethical_reasoning()

## 10. Interpretation, strengths, and limitations

### What this example demonstrates

- A constructive first-fit procedure can create pickup-and-delivery routes while explicitly recording vehicle assignments, depot legs, request coverage, precedence, and the aggregate capacity check.
- Ethical principles and stakeholder concerns can be represented as inspectable data structures and connected to clearly defined diagnostic indicators.
- A limited screening signal, explanatory notes, and a decision trail can make the evaluation easier to inspect.

### What the results do not establish

- The heuristic does not prove route optimality or compare its result with an optimum.
- Route distance is not a direct measure of customer welfare, driver working conditions, emissions, or business performance. The stakeholder scores are proxies defined for demonstration.
- The fairness indicator and route-distance screen cannot establish equal treatment or detect discrimination without suitable data, a defensible fairness definition, and a validated assessment process.
- Assumed privacy and accountability scores are not audits. An in-memory record is not proof of regulatory compliance or an immutable audit trail.
- The prototype does not resolve conflicts among principles, enforce real operational safety rules, or incorporate real stakeholder feedback.

### Appropriate use and natural extensions

This structure can support classroom discussion about value alignment, transparent assumptions, stakeholder trade-offs, and the difference between a proxy and an outcome. A real deployment would require domain-validated constraints, data governance, stakeholder participation, operational review, and evidence that each indicator measures the intended concept. Additional operational constraints or larger instances could be studied while retaining the same route-construction approach; this notebook does not implement those extensions.

In [ ]:
def final_summary(analysis: Dict[str, Any], optimizer: EthicalVRPPDOptimizer) -> None:
    """Summarize the measured indicators and the boundaries of the example."""
    print("\n" + "=" * 68)
    print("VRPPD ETHICAL DECISION-SUPPORT SUMMARY")
    print("=" * 68)
    print("\nRoute construction")
    print("- Method: request-order, first-fit assignment to the first vehicle with capacity.")
    print("- The constructed plan is checked for request coverage, precedence, capacity, and distance consistency.")
    print("- The heuristic does not establish global optimality.")

    print("\nDiagnostic indicators")
    print(f"- Recorded evaluations: {analysis['total_decisions']}")
    print(f"- Mean principle indicator: {np.mean(list(analysis['avg_principle_compliance'].values())):.2f}")
    print(f"- Mean stakeholder proxy: {np.mean(list(analysis['avg_stakeholder_impact'].values())):.2f}")
    print(f"- Mean transparency indicator: {analysis['avg_transparency']:.2f}")
    print(f"- Composite proxy: {analysis['avg_ethical_score']:.2f}")
    print(f"- Screening signals recorded: {analysis['total_biases_detected']}")
    print(f"- Mitigation actions recorded: {analysis['total_biases_mitigated']}")
    print(f"- Running proxy composite: {optimizer.performance_metrics['ethical_effectiveness']:.2f}")

    print("\nInterpretation limits")
    print("- Scores are illustrative proxies, not validated measures of ethical quality.")
    print("- Simulated feedback is not evidence from real stakeholders.")
    print("- No route modification, legal compliance certification, or bias-free guarantee is provided.")
    print("- Review assumptions and context before drawing conclusions from any indicator.")
    print("=" * 68)


final_summary(ethical_analysis, ethical_optimizer)